# Dwh NYC Yellow Taxi Payment Data

## Import Packages

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    TimestampType,
)

In [0]:
from pyspark import pipelines as dp

## Set Variables

## Create Schema

In [0]:
schema = StructType(
    [
        StructField(
            name="payment_type",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment":"Shows the RateCodeID"}
        ),
        StructField(
            name="payment_desc",
            dataType=StringType(),
            nullable=False,
            metadata={"comment":"Shows the Payment Code in Plain English"}
        ),
        StructField(
            name="sys_Insert_Dt",
            dataType=TimestampType(),
            nullable=False,
            metadata={"comment":"Shows the time the record was loaded"}
        ),
        StructField(
            name="sys_Insert_Fp",
            dataType=StringType(),
            nullable=False,
            metadata={"comment":"Shows the Location where the data came from"}
        ),
    ]
)

## ETL

In [0]:
@dp.temporary_view(name="dwh_nyc_taxi_payment_basis")
def bronze_dwh_taxi_zone_basis():
    df = spark.read.option("header", "true").csv("/Volumes/samples/databricks/datasets/nyctaxi/taxizone/taxi_payment_type.csv")
    df = df.withColumn("sys_Insert_Dt", F.current_timestamp())
    df = df.withColumn("sys_Insert_Fp", F.col("_metadata.file_path"))

    # Fit Dataframe to schema
    schema_columns = [(field.name, field.dataType) for field in schema.fields]
    df = df.select([F.col(col_name).cast(col_dtype) for col_name, col_dtype in schema_columns])
    return df

dp.create_streaming_table("analytics.bronze.dwh_nyc_taxi_payment", comment="This table shows the Payment Information of NYC", schema=schema)

dp.create_auto_cdc_from_snapshot_flow(
    target="analytics.bronze.dwh_nyc_taxi_payment",
    source="dwh_nyc_taxi_payment_basis",
    keys=["payment_type"],
    stored_as_scd_type=1,
)